In [1]:
from pathlib import Path
import sys

import pandas as pd


PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = (
        PROJECT_ROOT.parent
    )

sys.path.insert(
    0,
    str(PROJECT_ROOT)
)

In [2]:
from src.replay import TemporalReplay

from src.persistence import (
    buscar_run_replay,
    contar_transacoes,
    buscar_ultimas_transacoes
)

In [3]:
replay = TemporalReplay(
    intervalo_segundos=0.2
)

In [4]:
import pandas as pd


DATA_PATH = (
    PROJECT_ROOT
    / "raw"
    / "fraudTrain.csv"
)


dtypes = {
    "cc_num": "string",
    "trans_num": "string",
    "zip": "string",
    "merchant": "string",
    "category": "category",
    "gender": "category",
    "state": "category",
    "first": "string",
    "last": "string",
    "street": "string",
    "city": "string",
    "job": "string"
}


df = pd.read_csv(
    DATA_PATH,
    dtype=dtypes,
    parse_dates=[
        "trans_date_trans_time",
        "dob"
    ]
)

In [5]:
colunas_indice = [
    coluna
    for coluna in df.columns
    if str(coluna).startswith("Unnamed:")
]

if colunas_indice:
    df = df.drop(
        columns=colunas_indice
    )

In [6]:
df = (
    df
    .sort_values(
        "trans_date_trans_time"
    )
    .reset_index(drop=True)
)

In [7]:
n_total = len(df)

fim_treino = int(
    n_total * 0.70
)

fim_validacao = int(
    n_total * 0.85
)

df_teste = (
    df.iloc[
        fim_validacao:
    ]
    .copy()
)

In [8]:
resultado_replay = (
    replay.executar(
        transacoes=df_teste,
        run_id="replay_teste_20_v1",
        limite=20,
        atualizar_estado_a_cada=1
    )
)

resultado_replay

{'run_id': 'replay_teste_20_v1',
 'status': 'COMPLETED',
 'total_transacoes': 20,
 'processadas': 20,
 'intervalo_segundos': 0.2,
 'tempo_total_segundos': 4.488579208002193,
 'transacoes_por_segundo': 4.455752939447789}

In [9]:
buscar_run_replay(
    "replay_teste_20_v1"
)

{'run_id': 'replay_teste_20_v1',
 'status': 'COMPLETED',
 'total_transacoes': 20,
 'processadas': 20,
 'last_index': 19,
 'intervalo_segundos': 0.2,
 'started_at': '2026-08-17T01:36:56.676544+00:00',
 'updated_at': '2026-08-17T01:37:01.181271+00:00',
 'completed_at': '2026-08-17T01:37:01.181271+00:00',
 'error_message': None,
 'model_version': 'catboost_v1',
 'policy_version': 'decision_policy_v1'}

In [10]:
contar_transacoes(
    run_id="replay_teste_20_v1"
)

20

In [11]:
ultimas = (
    buscar_ultimas_transacoes(
        limite=5,
        run_id="replay_teste_20_v1"
    )
)

pd.DataFrame(
    ultimas
)[
    [
        "trans_date_trans_time",
        "score_fraude",
        "decisao",
        "is_fraud",
        "processed_at"
    ]
]

,trans_date_trans_time,score_fraude,decisao,is_fraud,processed_at
0,2020-04-03T18:08:15,0.000106,APROVAR,0,2026-08-17T01:37:01.173983+00:00
1,2020-04-03T18:06:27,0.000005,APROVAR,0,2026-08-17T01:37:00.947080+00:00
2,2020-04-03T18:05:56,0.000573,APROVAR,0,2026-08-17T01:37:00.701817+00:00
3,2020-04-03T18:04:43,0.000715,APROVAR,0,2026-08-17T01:37:00.459820+00:00
4,2020-04-03T18:01:17,0.000063,APROVAR,0,2026-08-17T01:37:00.224933+00:00
